# YOY-Based Forecast Model (Extensible Base-Year Actuals)

This notebook keeps the **forecasting logic unchanged**, but makes the FY base-year handling **extensible** as new quarters of actuals are appended to the input CSV.

**What’s now extensible (without changing model logic):**
- As new quarters for the base fiscal year (e.g., **FY26 Q2**) are added to the CSV, the model will automatically:
  - Treat those quarters as **actuals** (StdDev = 0)
  - Use remaining base-year quarters from `team_config` as **inputs**
  - Only append assumptions for **remaining** base-year quarters when computing rolling TTM YoY

Everything else (growth-rate generation, seasonality, jitter, StdDev scaling, export format) is preserved.


In [1]:
# ## 1) Imports & Setup
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [2]:
# ## 2) File Paths & Team Configuration
# Update these paths for your local environment.
INPUT_FILE = "/Users/matt.fritz/Desktop/SFDC Won Data FY16 pivot 260706.csv"
OUTPUT_FILE = "/Users/matt.fritz/Desktop/quarterly_forecasts_fy26_fy29_nomajor.csv"

# Optional: pin the model's base FY even if later FYs have 4 quarters in the CSV.
# Example: BASE_FY_OVERRIDE = "FY26"  (leave as None to keep auto-inference behavior)
BASE_FY_OVERRIDE = None

# When a CSV 'Forecast' overrides a model-generated forecast, keep a positive StdDev by
# scaling the model's coefficient of variation (StdDev/Forecast) onto the CSV value.
# This floor ensures StdDev remains > 0 after integer rounding in the export.
CSV_FORECAST_STDDEV_FLOOR = 1.0


# Team configurations:
# - ALL numeric revenue inputs have been removed; the CSV is now the single source of truth for base-year values.
# - Std dev scaling parameters retained (std_dev_multiplier, horizon_risk_rate, scaling_mode)
# - All other parameters retained exactly as before.
team_config = {
    'CorpFoundation': {
        # Growth rate parameters
        'decay_factor': 0.9,
        'volatility_dampen': 0.4,
        'growth_cap': None,
        'use_last_n_yoy': 24,
        # Std dev scaling
        'std_dev_multiplier': 0.4,
        'horizon_risk_rate': 0.9,
        'scaling_mode': 'exp',
        # Seasonality distribution (Q1-Q4)
        'seasonality': [0.23, 0.25, 0.26, 0.26]
    },
    'Marketplace': {
        'decay_factor': 0.5,
        'volatility_dampen': 0.1,
        'growth_cap': None,
        'use_last_n_yoy': 24,
        'std_dev_multiplier': 0.2,
        'horizon_risk_rate': 0.7,
        'scaling_mode': 'exp',
        'seasonality': [0.24, 0.25, 0.25, 0.26]
    },
    'Government': {
        'growth_cap': 80,
        'use_special_model': True,
        'std_dev_multiplier': 0.5,
        'horizon_risk_rate': 2.0,
        'scaling_mode': 'exp',
        'seasonality': [0.15, 0.20, 0.35, 0.30]
    }
}

# Forecast horizon (years beyond base FY). Logic unchanged elsewhere.
FORECAST_HORIZON_YEARS = 3

# These are inferred from the CSV at runtime (see infer_base_fiscal_year_from_csv).
BASE_FISCAL_YEAR = None
BASE_FY_LABEL = None
FORECAST_YEARS = None


In [3]:
# ## 3) Data Loading & Historical Analysis (logic unchanged; CSV is the single source of truth)

def load_historical_data(filepath):
    """
    Load historical data from CSV file.

    Expected columns (minimum):
      - Year, Quarter, FY
      - CorpFoundation, Government, MajorGifts, Marketplace

    Optional:
      - Type (e.g., Actual / Forecast). If present, Actual is preferred when duplicates exist.
    """
    try:
        df = pd.read_csv(filepath)
        df = prepare_input_df(df)
        print(f"Successfully loaded data from {filepath}")
        print(f"Shape: {df.shape}")
        return df
    except Exception as e:
        raise RuntimeError(f"Error loading file: {e}")


def prepare_input_df(df):
    """
    Minimal normalization for robustness:
      - If a 'Type' column exists and there are duplicates for the same FY/Year/Quarter,
        keep Actual over Forecast (matches the intended "latest truth" in the CSV).
    """
    df_out = df.copy()

    if 'Type' in df_out.columns:
        type_rank = (
            df_out['Type']
            .astype(str)
            .str.strip()
            .str.lower()
            .map({'actual': 0, 'forecast': 1})
            .fillna(2)
        )
        df_out = df_out.assign(_type_rank=type_rank)
        df_out = df_out.sort_values(['FY', 'Year', 'Quarter', '_type_rank'])
        df_out = df_out.drop_duplicates(subset=['FY', 'Year', 'Quarter'], keep='first')
        df_out = df_out.drop(columns=['_type_rank'])

    # Fold MajorGifts into CorpFoundation from the onset.
    # The MajorGifts flag is being retired; this revenue is booked as
    # CorpFoundation going forward, so treat the two as one channel everywhere.
    if 'MajorGifts' in df_out.columns:
        if 'CorpFoundation' in df_out.columns:
            df_out['CorpFoundation'] = (
                df_out['CorpFoundation'].fillna(0) + df_out['MajorGifts'].fillna(0)
            )
        else:
            df_out['CorpFoundation'] = df_out['MajorGifts']
        df_out = df_out.drop(columns=['MajorGifts'])

    return df_out


def parse_fy_label_to_year(fy_val):
    """
    Convert an FY label/value to a 4-digit year.
    Supports:
      - 'FY26' -> 2026
      - 26 -> 2026
      - 2026 -> 2026
    """
    if isinstance(fy_val, str):
        s = fy_val.strip().upper()
        if s.startswith('FY'):
            return 2000 + int(s[2:])
        # Fall back: numeric string
        if s.isdigit():
            n = int(s)
            return n if n >= 1000 else (2000 + n)
        raise ValueError(f"Unrecognized FY format: {fy_val}")

    if pd.isna(fy_val):
        raise ValueError("FY value is NaN/None")

    n = int(fy_val)
    return n if n >= 1000 else (2000 + n)


def infer_base_fiscal_year_from_csv(df, team_columns):
    """
    Infer the base fiscal year from the CSV by selecting the MOST RECENT FY that:
      - has at least 4 chronological quarters in that FY (after de-dup)
      - has non-null values for all specified team columns across those 4 quarters

    This keeps the model runnable when:
      - you add more forecast rows in future FYs (partial FYs are ignored)
      - you have no forecast rows (it will use the most recent complete FY of actuals)
    """
    if 'FY' not in df.columns:
        raise ValueError("CSV must contain an 'FY' column to infer the base fiscal year.")

    # Collect candidate FYs with parsed years
    fy_labels = df['FY'].dropna().unique().tolist()
    candidates = []
    for label in fy_labels:
        try:
            yr = parse_fy_label_to_year(label)
            candidates.append((yr, label))
        except Exception:
            continue

    # Sort from most recent to oldest
    candidates.sort(reverse=True, key=lambda x: x[0])

    for yr, label in candidates:
        grp = df.loc[df['FY'] == label].copy()
        if grp.empty:
            continue

        grp = grp.sort_values(['Year', 'Quarter'])
        if len(grp) < 4:
            continue

        # If extra rows exist, keep the most recent 4 within the FY
        grp4 = grp.tail(4).copy()

        # Require non-null values for each team column across all 4 quarters
        ok = True
        for col in team_columns:
            if col not in grp4.columns:
                ok = False
                break
            if grp4[col].isna().any():
                ok = False
                break

        if ok:
            return yr, str(label).strip().upper() if isinstance(label, str) else f"FY{str(yr)[-2:]}"

    # If no FY satisfies completeness, fail with a clear message
    raise ValueError(
        "Could not infer a complete base fiscal year from the CSV. "
        "Ensure there is at least one FY with 4 quarters and non-null values for all teams."
    )


def get_base_fy_rows_from_csv(df, base_fy_label, team_column):
    """
    Return two mappings for FY == base_fy_label:
      - amounts_by_fq: FiscalQuarter (1..4) -> amount
      - is_actual_by_fq: FiscalQuarter (1..4) -> bool (Type == 'Actual') if Type exists, else True

    FiscalQuarter is assigned by chronological order within the FY after sorting by (Year, Quarter).
    """
    if 'FY' not in df.columns:
        return {}, {}

    cols = ['Year', 'Quarter', team_column]
    has_type = 'Type' in df.columns
    if has_type:
        cols.insert(2, 'Type')

    df_fy = df.loc[df['FY'] == base_fy_label, cols].copy()
    if df_fy.empty:
        return {}, {}

    df_fy = df_fy.sort_values(['Year', 'Quarter']).reset_index(drop=True)

    # Keep the most recent 4 rows in-case there are extras
    if len(df_fy) >= 4:
        df_fy = df_fy.tail(4).reset_index(drop=True)

    df_fy['FiscalQuarter'] = np.arange(1, len(df_fy) + 1)

    amounts_by_fq = {int(fq): float(val) for fq, val in zip(df_fy['FiscalQuarter'], df_fy[team_column])}

    if has_type:
        is_actual_by_fq = {
            int(fq): (str(t).strip().lower() == 'actual')
            for fq, t in zip(df_fy['FiscalQuarter'], df_fy['Type'])
        }
    else:
        is_actual_by_fq = {int(fq): True for fq in df_fy['FiscalQuarter']}

    return amounts_by_fq, is_actual_by_fq


def build_base_year_plan(csv_amounts_by_fq, csv_is_actual_by_fq):
    """
    Determine the base fiscal year's quarter-by-quarter inputs strictly from the CSV.

    Returns:
      plan: dict quarter -> {'amount': float, 'is_actual': bool, 'source': str}
      fy_assumptions: dict quarter -> amount for quarters that are *forecast inputs*
                      (used for YoY assumption append; matches original Q2-Q4 behavior)
      base_total: float (sum of Q1-Q4 amounts per plan)
    """
    plan = {}
    fy_assumptions = {}
    base_total = 0.0

    for q in [1, 2, 3, 4]:
        if q not in csv_amounts_by_fq:
            raise KeyError(f"Missing required base-year value in CSV for fiscal Q{q} (need 4 quarters in the base FY).")

        amt = csv_amounts_by_fq[q]
        if pd.isna(amt):
            raise ValueError(f"Base-year value is NaN in CSV for fiscal Q{q}. Please provide a numeric value (use 0 if needed).")

        is_actual = bool(csv_is_actual_by_fq.get(q, True))
        plan[q] = {'amount': float(amt), 'is_actual': is_actual, 'source': 'csv'}

        # Match original behavior: only append Q2-Q4 assumptions into YoY calc,
        # and only when they are forecast inputs (not actuals).
        if (not is_actual) and (q in [2, 3, 4]):
            fy_assumptions[q] = float(amt)

        base_total += float(amt)

    return plan, fy_assumptions, base_total


def calculate_rolling_ttm_yoy(quarterly_data, team_column, fy_assumptions=None, assumption_year=2026):
    """
    Calculate rolling TTM YoY rates including base-year assumptions.

    TTM = Trailing Twelve Months (sum of 4 quarters)
    YoY = Current TTM / Year-ago TTM - 1
    """
    df_work = quarterly_data[['Year', 'Quarter', team_column]].copy()
    df_work = df_work.rename(columns={team_column: 'Amount'})

    # Append assumptions if provided (for more recent YoY points)
    if fy_assumptions:
        new_rows = []
        for q, amount in fy_assumptions.items():
            if q in [2, 3, 4]:  # Match original behavior: Q2-Q4 only
                new_rows.append({'Year': assumption_year, 'Quarter': q, 'Amount': amount})
        if new_rows:
            df_work = pd.concat([df_work, pd.DataFrame(new_rows)], ignore_index=True)

    # Sort by year and quarter
    df_work = df_work.sort_values(['Year', 'Quarter'])

    # Calculate TTM and YoY
    df_work['TTM'] = df_work['Amount'].rolling(window=4, min_periods=4).sum()
    df_work['TTM_YoY'] = (df_work['TTM'] / df_work['TTM'].shift(4) - 1) * 100

    yoy_rates = df_work['TTM_YoY'].dropna().values
    return yoy_rates, df_work


def calculate_historical_metrics(df, team_column, config, fy_assumptions=None, base_fy=2026):
    """
    Calculate historical YoY rates and quarterly volatility for a team.
    Includes base-year assumptions to get more recent YoY data points.

    NOTE: If a 'Type' column exists, historical metrics are calculated from Actual rows only
    (forecast rows are treated as base-year inputs via fy_assumptions, matching the original workflow).
    """
    df_hist = df
    if 'Type' in df.columns:
        df_hist = df.loc[df['Type'].astype(str).str.strip().str.lower() == 'actual'].copy()

    # If no assumptions provided, default to none (CSV is source of truth)
    if fy_assumptions is None:
        fy_assumptions = {}

    yoy_rates, ttm_df = calculate_rolling_ttm_yoy(
        df_hist, team_column, fy_assumptions=fy_assumptions, assumption_year=base_fy
    )

    # Quarterly coefficient of variation (excluding projections for volatility)
    historical_amounts = df_hist[team_column].dropna()

    if len(historical_amounts) >= 4:
        df_temp = df_hist[df_hist[team_column].notna()].copy()
        quarterly_stats = df_temp.groupby('Quarter')[team_column].agg(['mean', 'std'])
        quarterly_cv = (quarterly_stats['std'] / quarterly_stats['mean']).mean()
    else:
        quarterly_cv = 0.15  # fallback default if insufficient data

    # Print recent YoY rates (same behavior)
    last_n = min(16, len(yoy_rates))
    print(f"\n  YoY rates calculated: {len(yoy_rates)} total")
    if len(yoy_rates) > 0:
        print(f"  Most recent YoY rates: {[f'{x:.0f}%' for x in yoy_rates[-last_n:]]}")
        if fy_assumptions:
            print(f"  Note: Last 3 rates include {BASE_FY_LABEL} assumptions")

    return {
        'yoy_rates': yoy_rates,
        'quarterly_cv': quarterly_cv,
        'ttm_dataframe': ttm_df
    }


In [4]:
# ## 4) Standard Deviation Calculation (unchanged)
def calculate_quarterly_std(base_forecast, historical_cv, std_dev_multiplier=1.0,
                            time_years=0, horizon_risk_rate=0.2, scaling_mode='linear'):
    """
    Calculate quarterly standard deviation with flexible time scaling.

    scaling_mode ∈ {'linear', 'sqrt', 'exp'}.
    """
    if scaling_mode == 'linear':
        time_factor = 1 + (time_years * horizon_risk_rate)
    elif scaling_mode == 'sqrt':
        time_factor = np.sqrt(1 + time_years * horizon_risk_rate)
    elif scaling_mode == 'exp':
        time_factor = (1 + horizon_risk_rate) ** time_years
    else:
        raise ValueError(f"Unsupported scaling_mode: {scaling_mode}")

    std_dev = base_forecast * historical_cv * std_dev_multiplier * time_factor
    return std_dev


In [5]:
# ## 5) Core Forecasting Functions (unchanged)
def generate_growth_rate(historical_yoy, decay_factor, volatility_dampen, growth_cap=None, use_last_n_yoy=16):
    """Draw a single annual growth rate (FY+1..FY+3) around the weighted historical mean."""
    if len(historical_yoy) == 0:
        return np.random.normal(5, 10)

    # Use only most recent N YoY rates
    hist = historical_yoy[-use_last_n_yoy:] if len(historical_yoy) > use_last_n_yoy else historical_yoy

    # Exponential decay weights (recent points matter more)
    n = len(hist)
    weights = np.array([decay_factor ** (n - i - 1) for i in range(n)])
    weights /= weights.sum()

    weighted_mean = np.sum(hist * weights)
    historical_std = np.std(hist) if n > 1 else 15  # fallback if only one point

    # Random draw for annual growth (volatility_dampen shrinks the historical std)
    growth_rate = np.random.normal(weighted_mean, historical_std * volatility_dampen)

    # Optional growth cap
    if growth_cap is not None:
        growth_rate = min(growth_rate, growth_cap)

    return growth_rate


def generate_growth_rate_stochastic(historical_yoy, n_sim=10_000, summary='mean', cap=None):
    """Monte Carlo version returning summary of simulated YoY growths."""
    sims = []
    for _ in range(n_sim):
        if len(historical_yoy) < 3:
            val = np.random.choice(
                [-50, -20, 0, 30, 80, 150],
                p=[0.1, 0.2, 0.2, 0.2, 0.2, 0.1]
            )
        else:
            p20, p50, p80 = np.percentile(historical_yoy, [20, 50, 80])
            rand = np.random.random()
            if rand < 0.2:
                val = np.random.uniform(min(p20, -50), p50)
            elif rand < 0.8:
                val = np.random.uniform(0, p80)
            else:
                val = np.random.uniform(p80, min(200, p80 * 1.5))
        sims.append(val)

    sims = np.array(sims)
    if summary == 'mean':
        result = sims.mean()
    elif summary in ['median', 'p50']:
        result = np.percentile(sims, 50)
    elif summary == 'p90':
        result = np.percentile(sims, 90)
    elif summary == 'p10':
        result = np.percentile(sims, 10)
    else:
        raise ValueError(f"Unsupported summary type: {summary}")

    if cap is not None:
        result = min(result, cap)

    return result


In [6]:
# ## 6) Forecast Generation (base-year now sourced strictly from CSV; future-year logic unchanged)

def generate_quarterly_forecasts(team_name, config, historical_metrics, base_year_plan, base_fy):
    """Generate quarterly forecasts through FY+3, including base-year actuals/inputs."""
    forecasts = []

    # Extract parameters
    seasonality = config['seasonality']
    historical_yoy = historical_metrics['yoy_rates']
    quarterly_cv = historical_metrics['quarterly_cv']

    # --- Base-year quarters (Q1-Q4): values come strictly from CSV; Actual vs Forecast uses CSV 'Type' ---
    for quarter in [1, 2, 3, 4]:
        amt = base_year_plan[quarter]['amount']
        is_actual = base_year_plan[quarter]['is_actual']

        if is_actual:
            forecasts.append({
                'Team': team_name,
                'FiscalYear': base_fy,
                'Quarter': quarter,
                'Forecast': amt,
                'StdDev': 0,
                'AnnualGrowth': 0
            })
        else:
            std_dev = calculate_quarterly_std(
                amt,
                quarterly_cv,
                std_dev_multiplier=config.get('std_dev_multiplier', 1.0),
                time_years=0,  # base FY
                horizon_risk_rate=config.get('horizon_risk_rate', 0.2),
                scaling_mode=config.get('scaling_mode', 'linear')
            )
            forecasts.append({
                'Team': team_name,
                'FiscalYear': base_fy,
                'Quarter': quarter,
                'Forecast': amt,
                'StdDev': std_dev
            })

    # Base-year total from actuals + remaining inputs (same concept as before)
    base_total = sum(base_year_plan[q]['amount'] for q in [1, 2, 3, 4])

    # Use the base year's actual quarter shape for future years instead of the flat config array
    if base_total > 0:
        seasonality = [base_year_plan[q]['amount'] / base_total for q in [1, 2, 3, 4]]

    # --- Generate growth rates for FY+1..FY+3 (unchanged) ---
    annual_growth_rates = []
    for _ in range(3):
        if config.get('use_special_model'):
            growth_rate = generate_growth_rate_stochastic(
                historical_yoy,
                n_sim=10_000,
                summary='mean',
                cap=config.get('growth_cap')
            )
        else:
            growth_rate = generate_growth_rate(
                historical_yoy,
                config['decay_factor'],
                config['volatility_dampen'],
                config.get('growth_cap'),
                config.get('use_last_n_yoy', 16)
            )
        annual_growth_rates.append(growth_rate)

    # --- Apply growth and generate quarterly forecasts (unchanged) ---
    annual_base = base_total
    for year_idx, fiscal_year in enumerate(FORECAST_YEARS):
        annual_base = annual_base * (1 + annual_growth_rates[year_idx] / 100)

        for quarter in range(1, 5):
            forecast_amt = annual_base * seasonality[quarter - 1]

            # Add small random variation for realism (unchanged)
            forecast_amt *= np.random.normal(1, 0.02)

            std_dev = calculate_quarterly_std(
                forecast_amt,
                quarterly_cv,
                std_dev_multiplier=config.get('std_dev_multiplier', 1.0),
                time_years=fiscal_year - base_fy,
                horizon_risk_rate=config.get('horizon_risk_rate', 0.2),
                scaling_mode=config.get('scaling_mode', 'linear')
            )

            forecasts.append({
                'Team': team_name,
                'FiscalYear': fiscal_year,
                'Quarter': quarter,
                'Forecast': forecast_amt,
                'StdDev': std_dev,
                'AnnualGrowth': annual_growth_rates[year_idx]
            })

    return pd.DataFrame(forecasts), annual_growth_rates


# --- CSV overrides (post-model overlay; does not change forecasting logic) ---

def extract_csv_overrides(df, column_mapping, min_fiscal_year=None):
    """
    Convert wide CSV rows into long-form overrides keyed by (Team, FiscalYear, Quarter).

    - Uses the same fiscal-quarter assignment as the base-year plan:
      for each FY label, sort by (Year, Quarter), keep the most recent 4 rows,
      then assign FiscalQuarter = 1..N in that order.

    - Filters to fiscal years >= min_fiscal_year when provided.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=['Team', 'FiscalYear', 'Quarter', 'CSVValue', 'CSVType'])

    if 'FY' not in df.columns:
        return pd.DataFrame(columns=['Team', 'FiscalYear', 'Quarter', 'CSVValue', 'CSVType'])

    has_type = 'Type' in df.columns
    records = []

    # Group by FY label (e.g., FY26)
    df_fy = df.dropna(subset=['FY']).copy()
    for fy_label, grp in df_fy.groupby('FY'):
        try:
            fy_year = parse_fy_label_to_year(fy_label)
        except Exception:
            continue

        if (min_fiscal_year is not None) and (fy_year < min_fiscal_year):
            continue

        grp = grp.sort_values(['Year', 'Quarter']).copy()
        if len(grp) > 4:
            grp = grp.tail(4).copy()

        grp = grp.reset_index(drop=True)
        grp['FiscalQuarter'] = np.arange(1, len(grp) + 1)

        # For each quarter row, capture team values
        for _, r in grp.iterrows():
            fq = int(r['FiscalQuarter'])
            csv_type = str(r['Type']).strip().lower() if has_type else 'actual'

            for team_name, col in column_mapping.items():
                if col not in grp.columns:
                    continue
                val = r.get(col, np.nan)
                if pd.isna(val):
                    continue

                records.append({
                    'Team': team_name,
                    'FiscalYear': int(fy_year),
                    'Quarter': fq,
                    'CSVValue': float(val),
                    'CSVType': csv_type
                })

    if not records:
        return pd.DataFrame(columns=['Team', 'FiscalYear', 'Quarter', 'CSVValue', 'CSVType'])

    overrides = pd.DataFrame(records)

    # If duplicates exist for a (Team,Year,Quarter), keep the last occurrence
    overrides = (
        overrides
        .sort_values(['Team', 'FiscalYear', 'Quarter'])
        .drop_duplicates(subset=['Team', 'FiscalYear', 'Quarter'], keep='last')
        .reset_index(drop=True)
    )

    return overrides


def apply_csv_overrides_to_forecast_df(forecast_df, df_csv, column_mapping, base_fiscal_year, stddev_floor=1.0):
    """
    Option A overlay:
      - Run the model as-is.
      - Then overwrite the model output with any Actual/Forecast values present in the CSV
        from base_fiscal_year onward.
      - Actual overrides -> StdDev = 0
      - Forecast overrides -> keep positive StdDev by scaling the model's CV onto the CSV value:
            StdDev_new = max(stddev_floor, (StdDev_model/Forecast_model) * Forecast_csv)
        If the model does not have that period (CSV extends beyond horizon), use the team's median model CV.
    """
    csv_overrides = extract_csv_overrides(df_csv, column_mapping, min_fiscal_year=base_fiscal_year)
    if csv_overrides.empty:
        return forecast_df

    # Team-level median CV from model output (fallback for beyond-horizon overrides)
    cv_src = forecast_df.loc[
        (forecast_df['Forecast'] > 0) & (forecast_df['StdDev'] > 0),
        ['Team', 'Forecast', 'StdDev']
    ].copy()

    if len(cv_src) > 0:
        cv_src['CV'] = cv_src['StdDev'] / cv_src['Forecast']
        team_median_cv = cv_src.groupby('Team')['CV'].median().to_dict()
    else:
        team_median_cv = {}

    DEFAULT_CV = 0.15

    # Outer merge so CSV can extend beyond model horizon (restricted to >= base FY)
    merged = forecast_df.merge(
        csv_overrides,
        on=['Team', 'FiscalYear', 'Quarter'],
        how='outer'
    )

    # Compute row-level model CV where available
    merged['ModelCV'] = np.where(
        (merged['Forecast'] > 0) & (merged['StdDev'] > 0),
        merged['StdDev'] / merged['Forecast'],
        np.nan
    )
    merged['TeamCV'] = merged['Team'].map(team_median_cv).fillna(DEFAULT_CV)
    cv_used = merged['ModelCV'].fillna(merged['TeamCV'])

    has_override = merged['CSVValue'].notna()

    if 'CSVType' in merged.columns:
        csv_type = (
            merged['CSVType']
            .fillna('actual')
            .astype(str)
            .str.strip()
            .str.lower()
        )
    else:
        csv_type = pd.Series(['actual'] * len(merged), index=merged.index)

    is_actual = has_override & (csv_type == 'actual')
    is_forecast = has_override & (csv_type != 'actual')

    # Apply actual overrides
    merged.loc[is_actual, 'Forecast'] = merged.loc[is_actual, 'CSVValue']
    merged.loc[is_actual, 'StdDev'] = 0.0

    # Apply forecast overrides (positive StdDev)
    merged.loc[is_forecast, 'Forecast'] = merged.loc[is_forecast, 'CSVValue']
    merged.loc[is_forecast, 'StdDev'] = np.maximum(
        float(stddev_floor),
        (cv_used.loc[is_forecast] * merged.loc[is_forecast, 'CSVValue']).astype(float)
    )

    # Clean up helper cols
    merged = merged.drop(columns=[c for c in ['CSVValue', 'CSVType', 'ModelCV', 'TeamCV'] if c in merged.columns])

    merged = merged.sort_values(['Team', 'FiscalYear', 'Quarter']).reset_index(drop=True)

    return merged

def run_forecast_model():
    """Run the complete forecasting model and return the full forecast dataframe."""
    print("="*80)
    print("YOY-BASED QUARTERLY FORECASTING MODEL (Updated Std Dev Scaling)")
    print("="*80)
    print(f"\nInput file: {INPUT_FILE}")
    print(f"Output file: {OUTPUT_FILE}")

    # Load historical data
    print("\nLoading historical data...")
    df = load_historical_data(INPUT_FILE)

    # Infer base FY from CSV (single source of truth)
    global BASE_FISCAL_YEAR, BASE_FY_LABEL, FORECAST_YEARS
    column_mapping = {
        'CorpFoundation': 'CorpFoundation',
        'Marketplace': 'Marketplace',
        'Government': 'Government'
    }
    team_columns = list(column_mapping.values())

    # If provided, use a fixed base FY (prevents base-year shifting when future FY forecasts are added)
    if BASE_FY_OVERRIDE is not None:
        BASE_FY_LABEL = str(BASE_FY_OVERRIDE).strip().upper()
        BASE_FISCAL_YEAR = parse_fy_label_to_year(BASE_FY_LABEL)
    else:
        BASE_FISCAL_YEAR, BASE_FY_LABEL = infer_base_fiscal_year_from_csv(df, team_columns)
    FORECAST_YEARS = [BASE_FISCAL_YEAR + 1, BASE_FISCAL_YEAR + 2, BASE_FISCAL_YEAR + 3]

    print(f"\nBase fiscal year inferred from CSV: {BASE_FY_LABEL}")
    print(f"Forecast years: FY{str(FORECAST_YEARS[0])[-2:]}-FY{str(FORECAST_YEARS[-1])[-2:]}")

    all_forecasts = []

    for team_name, config in team_config.items():
        print(f"\n{team_name}")
        print("-"*60)

        team_column = column_mapping[team_name]

        # Pull base-year values from the CSV for this team (FY == BASE_FY_LABEL)
        csv_amounts_by_fq, csv_is_actual_by_fq = get_base_fy_rows_from_csv(df, BASE_FY_LABEL, team_column)

        # Resolve base-year plan (strictly from CSV)
        base_plan, fy_assumptions, _base_total = build_base_year_plan(csv_amounts_by_fq, csv_is_actual_by_fq)

        # Historical metrics including only base-year forecast assumptions (Q2-Q4) when present
        historical_metrics = calculate_historical_metrics(
            df,
            team_column,
            config,
            fy_assumptions=fy_assumptions,
            base_fy=BASE_FISCAL_YEAR
        )
        print(f"  Historical quarterly CV: {historical_metrics['quarterly_cv']:.1%}")

        # Show base-year snapshot (CSV-driven)
        print(f"\n  {BASE_FY_LABEL} Inputs:")
        for q in [1, 2, 3, 4]:
            label = "actual" if base_plan[q]['is_actual'] else "input"
            amt = base_plan[q]['amount']
            print(f"    Q{q} ({label}): ${amt/1e6:.1f}M")

        # Generate all forecasts (base FY + FY+1..FY+3)
        team_forecasts, growth_rates = generate_quarterly_forecasts(
            team_name,
            config,
            historical_metrics,
            base_plan,
            BASE_FISCAL_YEAR
        )
        print(f"\n  Generated growth rates for FY{str(FORECAST_YEARS[0])[-2:]}-{str(FORECAST_YEARS[-1])[-2:]}: {[f'{x:.0f}%' for x in growth_rates]}")

        # Weighted mean for reference (guarded; unchanged when present)
        if len(historical_metrics['yoy_rates']) > 0 and 'decay_factor' in config:
            n = len(historical_metrics['yoy_rates'])
            weights = np.array([config['decay_factor'] ** (n - i - 1) for i in range(n)])
            weights = weights / weights.sum()
            weighted_mean = np.sum(historical_metrics['yoy_rates'] * weights)
            print(f"  Weighted historical YoY mean: {weighted_mean:.1f}%")

        all_forecasts.append(team_forecasts)

    # Combine all forecasts
    forecast_df = pd.concat(all_forecasts, ignore_index=True)
    forecast_df = forecast_df.sort_values(['Team', 'FiscalYear', 'Quarter'])

    # Overlay any CSV-provided Actuals/Forecasts onto the model output (post-model; Option A)
    forecast_df = apply_csv_overrides_to_forecast_df(
        forecast_df,
        df,
        column_mapping,
        BASE_FISCAL_YEAR,
        stddev_floor=CSV_FORECAST_STDDEV_FLOOR
    )
    forecast_df = forecast_df.sort_values(['Team', 'FiscalYear', 'Quarter'])


    # Display summary (same logic, but uses inferred BASE_FISCAL_YEAR)
    print("\n" + "="*80)
    print("FORECAST SUMMARY")
    print("="*80)

    for team in team_config.keys():
        team_df = forecast_df[forecast_df['Team'] == team]
        print(f"\n{team}")
        print("-"*40)
        print(f"  {BASE_FY_LABEL} (values from CSV; forecast inputs labeled as 'input'):")
        for q in range(1, 5):
            row = team_df[(team_df['FiscalYear'] == BASE_FISCAL_YEAR) & (team_df['Quarter'] == q)]
            if len(row) > 0:
                row = row.iloc[0]
                print(f"    Q{q}: ${row['Forecast']/1e6:6.1f}M")

        for fy in [FORECAST_YEARS[0], FORECAST_YEARS[-1]]:
            fy_total = team_df[team_df['FiscalYear'] == fy]['Forecast'].sum()
            print(f"  FY{str(fy)[-2:]} Total: ${fy_total/1e6:.1f}M")

        fy_last_total = team_df[team_df['FiscalYear'] == FORECAST_YEARS[-1]]['Forecast'].sum()
        fy_base_total = team_df[team_df['FiscalYear'] == BASE_FISCAL_YEAR]['Forecast'].sum()
        cagr = (fy_last_total / fy_base_total) ** (1/3) - 1
        print(f"  CAGR FY{str(BASE_FISCAL_YEAR)[-2:]}-{str(FORECAST_YEARS[-1])[-2:]}: {cagr*100:.1f}%")

    # Export (unchanged format)
    export_df = forecast_df[['Team', 'FiscalYear', 'Quarter', 'Forecast', 'StdDev']].copy()
    export_df['Forecast'] = export_df['Forecast'].round(0).astype(int)
    export_df['StdDev'] = export_df['StdDev'].round(0).astype(int)

    export_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nForecasts exported to: {OUTPUT_FILE}")

    return forecast_df


In [7]:
# ## 7) Pivot Tables (unchanged)

def build_pivots(forecast_df):
    # Pivot 1: Forecast values by FY/Quarter
    pivot_forecast = (
        forecast_df
        .pivot_table(index=['FiscalYear', 'Quarter'], columns='Team', values='Forecast', aggfunc='sum')
        .sort_index()
    )

    # Pivot 2: Annual totals
    pivot_annual = (
        forecast_df
        .pivot_table(index='FiscalYear', columns='Team', values='Forecast', aggfunc='sum')
        .sort_index()
    )

    # Pivot 3: Coefficient of Variation (CV = StdDev / Forecast)
    pivot_cv = (
        forecast_df
        .assign(CV=lambda x: np.where(x['Forecast'] != 0, x['StdDev'] / x['Forecast'], np.nan))
        .pivot_table(
            index=['FiscalYear', 'Quarter'],
            columns='Team',
            values='CV',
            aggfunc='mean'
        )
        .sort_index()
    )
    pivot_cv = pivot_cv * 100  # percent

    # Optional: make column order consistent with config definition
    team_order = list(team_config.keys())
    pivot_forecast = pivot_forecast[team_order]
    pivot_annual = pivot_annual[team_order]
    pivot_cv = pivot_cv[team_order]

    print("\n=== Forecasts by Fiscal Year / Quarter ===")
    print(pivot_forecast.round(0).astype(int))

    print("\n=== Annual Forecast Totals (Sum of Forecast) ===")
    print(pivot_annual.round(0).astype(int))

    print("\n=== Coefficient of Variation (CV %) by Fiscal Year / Quarter ===")
    print(pivot_cv.round(1))


In [8]:
# ## 8) Execute end-to-end (unchanged)
try:
    np.random.seed(45)
    forecast_df = run_forecast_model()
    build_pivots(forecast_df)
    print("\nRun complete.")
except Exception as e:
    print(f"\n[ERROR] {e}\n")
    print("Tip: Check INPUT_FILE path in the config cell and ensure the CSV exists at that location.")


YOY-BASED QUARTERLY FORECASTING MODEL (Updated Std Dev Scaling)

Input file: /Users/matt.fritz/Desktop/SFDC Won Data FY16 pivot 260706.csv
Output file: /Users/matt.fritz/Desktop/quarterly_forecasts_fy26_fy29_nomajor.csv

Loading historical data...
Successfully loaded data from /Users/matt.fritz/Desktop/SFDC Won Data FY16 pivot 260706.csv
Shape: (44, 7)

Base fiscal year inferred from CSV: FY26
Forecast years: FY27-FY29

CorpFoundation
------------------------------------------------------------

  YoY rates calculated: 37 total
  Most recent YoY rates: ['-2%', '-24%', '-15%', '-1%', '-24%', '-7%', '-8%', '-15%', '-2%', '-19%', '-19%', '1%', '21%', '30%', '33%', '-4%']
  Historical quarterly CV: 39.8%

  FY26 Inputs:
    Q1 (actual): $21.2M
    Q2 (actual): $9.5M
    Q3 (actual): $13.0M
    Q4 (actual): $17.4M

  Generated growth rates for FY27-29: ['4%', '5%', '1%']
  Weighted historical YoY mean: 3.2%

Marketplace
------------------------------------------------------------

  YoY rat

In [9]:
# ============================================================================
# BACKTEST — does rolling MajorGifts into CorpFoundation hurt the combined forecast?
# Rolling-origin + Monte Carlo, reusing the notebook's own forecasting functions.
# separate: forecast Corp (corp model) + Major (its special capped/percentile model),
#           each with its own seasonality, then sum.
# merged:   forecast ONE (Corp+Major) series with CorpFoundation's model + seasonality.
# Both scored against the SAME actual Corp+Major total in each held-out year.
# Bypasses the CSV-override overlay so held-out actuals don't leak back onto the forecast.
#
# KNOBS
CORP_COL, MAJOR_COL  = "CorpFoundation", "MajorGifts"
MERGED_CONFIG_SOURCE = "CorpFoundation"  # config the merged line is modeled with (your hypothesis)
K_REPS               = 200               # MC reps per config per holdout (raise for tighter est.)
MIN_HISTORY_FYS      = 1                 # complete FYs required up to & incl. the base year
SEED                 = 45
# ============================================================================
import io, contextlib, numpy as np, pandas as pd

_df = load_historical_data(INPUT_FILE)
if "Type" in _df.columns:
    _df = _df[_df["Type"].astype(str).str.strip().str.lower() == "actual"].copy()
def _yr(l):
    try: return parse_fy_label_to_year(l)
    except Exception: return np.nan
_df["_fyy"] = _df["FY"].map(_yr)
_df = _df.dropna(subset=["_fyy"]); _df["_fyy"] = _df["_fyy"].astype(int)

def _complete_fys(frame):
    out = {}
    for l in frame["FY"].dropna().unique():
        y = _yr(l)
        if np.isnan(y): continue
        g = frame[frame["FY"] == l].sort_values(["Year", "Quarter"]).tail(4)
        if len(g) >= 4 and not g[[CORP_COL, MAJOR_COL]].isna().any().any():
            out[int(y)] = str(l)
    return dict(sorted(out.items()))
complete   = _complete_fys(_df)
comp_years = sorted(complete)
holdouts   = [(h, complete[h]) for h in comp_years
              if (h - 1) in complete and sum(y <= h - 1 for y in comp_years) >= MIN_HISTORY_FYS]
if not holdouts:
    raise ValueError("No holdout year has enough complete-FY history. Lower MIN_HISTORY_FYS.")

cfg_corp, cfg_major = team_config["CorpFoundation"], team_config["MajorGifts"]
cfg_merged          = team_config[MERGED_CONFIG_SOURCE]

def _fc_one(df_hist, team_col, config, base_year, base_lbl):
    """base+1 (first-forecast-year) total for one channel via the notebook's own pipeline."""
    global FORECAST_YEARS, BASE_FY_LABEL
    FORECAST_YEARS, BASE_FY_LABEL = [base_year+1, base_year+2, base_year+3], base_lbl
    amt, act = get_base_fy_rows_from_csv(df_hist, base_lbl, team_col)
    plan, assump, _ = build_base_year_plan(amt, act)
    with contextlib.redirect_stdout(io.StringIO()):
        hm = calculate_historical_metrics(df_hist, team_col, config, fy_assumptions=assump, base_fy=base_year)
        fdf, _ = generate_quarterly_forecasts(team_col, config, hm, plan, base_year)
    return fdf.loc[fdf["FiscalYear"] == base_year + 1, "Forecast"].sum()

rows = []
for h_year, h_lbl in holdouts:
    base_year, base_lbl = h_year - 1, complete[h_year - 1]
    hist = _df[_df["_fyy"] <= base_year].copy()
    hist["_CorpMajor"] = hist[CORP_COL] + hist[MAJOR_COL]
    g = _df[_df["FY"] == h_lbl].sort_values(["Year", "Quarter"]).tail(4)
    actual = float(g[CORP_COL].sum() + g[MAJOR_COL].sum())

    sep_p, mrg_p = [], []
    for r in range(K_REPS):
        np.random.seed(SEED + r)
        sep_p.append(_fc_one(hist, CORP_COL, cfg_corp, base_year, base_lbl)
                     + _fc_one(hist, MAJOR_COL, cfg_major, base_year, base_lbl))
        np.random.seed(SEED + r)
        mrg_p.append(_fc_one(hist, "_CorpMajor", cfg_merged, base_year, base_lbl))
    sep_p, mrg_p = np.array(sep_p), np.array(mrg_p)
    rows.append({
        "holdout": h_lbl, "actual_$M": actual/1e6,
        "sep_pred_$M": sep_p.mean()/1e6, "mrg_pred_$M": mrg_p.mean()/1e6,
        "separate_MAPE%": np.abs(sep_p - actual).mean()/actual*100,
        "merged_MAPE%":   np.abs(mrg_p - actual).mean()/actual*100,
    })

bt = pd.DataFrame(rows)
bt["Δ(mrg-sep)"] = bt["merged_MAPE%"] - bt["separate_MAPE%"]
pd.set_option("display.width", 200, "display.max_columns", 20)
print(f"Merged line modeled as: {MERGED_CONFIG_SOURCE} | reps/yr: {K_REPS} | holdouts: {len(bt)}\n")
print(bt.round(2).to_string(index=False))
d = bt["Δ(mrg-sep)"].mean()
print(f"\nMean 1-yr-ahead MAPE — separate {bt['separate_MAPE%'].mean():.2f}%  |  "
      f"merged {bt['merged_MAPE%'].mean():.2f}%  |  Δ {d:+.2f} pts "
      f"({'merge worse' if d > 0 else 'merge same/better'})")

Successfully loaded data from /Users/matt.fritz/Desktop/SFDC Won Data FY16 pivot 260706.csv
Shape: (44, 7)


KeyError: "['MajorGifts'] not in index"

In [ ]:
# ============================================================================
# EVIDENCE CHECK — (a) is "separate MajorGifts" on real signal or a fallback prior?
#                  (b) does Major actually behave differently from CorpFoundation?
#                  (c) how much can a Major miss move the combined line?
# Uses only FY25-FY26 actuals (the only years Major exists as its own channel).
# ============================================================================
import io, contextlib, numpy as np, pandas as pd
cfg_corp, cfg_major = team_config["CorpFoundation"], team_config["MajorGifts"]

# --- (a) fallback-path check at the FY25 base -------------------------------
hist25 = _df[_df["_fyy"] <= 2025].copy()
def _yoy(col, cfg):
    with contextlib.redirect_stdout(io.StringIO()):
        return calculate_historical_metrics(hist25, col, cfg, fy_assumptions={}, base_fy=2025)["yoy_rates"]
maj_yoy, corp_yoy = _yoy(MAJOR_COL, cfg_major), _yoy(CORP_COL, cfg_corp)
fb_vals, fb_p = np.array([-50,-20,0,30,80,150]), np.array([.1,.2,.2,.2,.2,.1])
fb_mean = float((fb_vals*fb_p).sum())
print("=== (a) MajorGifts growth signal at the FY25 base ===")
print(f"  MajorGifts usable YoY points: {len(maj_yoy)}  -> "
      f"{'FALLBACK (len<3): growth is a data-free prior' if len(maj_yoy)<3 else 'real signal'}")
print(f"    fallback draws ~{fb_mean:.0f}% mean growth (cap {cfg_major.get('growth_cap')}%), "
      f"independent of Major's actuals")
print(f"  CorpFoundation usable YoY points: {len(corp_yoy)}  (weights up to last "
      f"{cfg_corp.get('use_last_n_yoy')})")

# --- (b) Corp vs Major side by side -----------------------------------------
lbl = {y: _df.loc[_df["_fyy"] == y, "FY"].iloc[0] for y in (2025, 2026)}
def qv(y, col): 
    g = _df[_df["FY"] == lbl[y]].sort_values(["Year","Quarter"]).tail(4)
    return g[col].to_numpy(dtype=float)

lev = []
for y in (2025, 2026):
    for col in (CORP_COL, MAJOR_COL):
        v = qv(y, col)
        lev.append({"FY": lbl[y], "channel": col, "annual_$M": v.sum()/1e6,
                    **{f"Q{i+1}_$M": v[i]/1e6 for i in range(4)}})
print("\n=== (b) Levels ($M) ===")
print(pd.DataFrame(lev).round(2).to_string(index=False))

print("\n=== Quarterly share of own annual (%) — the 'rhythm' comparison ===")
sh = []
for y in (2025, 2026):
    for col in (CORP_COL, MAJOR_COL):
        v = qv(y, col); s = v/v.sum()*100
        sh.append({"FY": lbl[y], "channel": col, **{f"Q{i+1}%": s[i] for i in range(4)}})
for col, cfg in ((CORP_COL, cfg_corp), (MAJOR_COL, cfg_major)):
    sh.append({"FY":"config","channel":col, **{f"Q{i+1}%": cfg["seasonality"][i]*100 for i in range(4)}})
print(pd.DataFrame(sh).round(1).to_string(index=False))

print("\n=== The one YoY point you have (FY26 vs FY25) ===")
for col in (CORP_COL, MAJOR_COL):
    a25, a26 = qv(2025, col).sum(), qv(2026, col).sum()
    print(f"  {col:16s}: {a25/1e6:6.1f}M -> {a26/1e6:6.1f}M   YoY {(a26/a25-1)*100:+.1f}%")

# --- (c) exposure sizing ----------------------------------------------------
c26, m26 = qv(2026, CORP_COL).sum(), qv(2026, MAJOR_COL).sum()
share = m26/(c26+m26)*100
print("\n=== (c) Cost-of-being-wrong sizing (FY26) ===")
print(f"  MajorGifts = {share:.1f}% of combined Corp+Major (${m26/1e6:.1f}M of ${(c26+m26)/1e6:.1f}M)")
for miss in (20, 40, 60):
    print(f"    {miss}% mis-forecast of Major = {miss*share/100:.1f} pts of combined-line error")